# One-Phonon Diffuse Scattering — Rigid-Body Crystal (6O2H, P1)

One rigid body per unit cell (whole lysozyme molecule), 6 DOF: $(\Omega, v)$.

**Pipeline:** hybrid density → interface detection → K-matrix input →
dynamical matrix → diffuse map → band structure → in-notebook surface animation.


In [ ]:
import numpy as np
import gemmi
import pathlib
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.linalg import eigh
from itertools import product

np.set_printoptions(precision=4, suppress=True)


In [ ]:
PDB_FILE = pathlib.Path("6o2h.pdb")
SF_CIF   = pathlib.Path("6o2h-sf.cif")

D_MIN_MODEL    = 1.5    # Å — resolution for model density (phases)
RATE_MODEL     = 1.5
CONTACT_CUTOFF = 5.0    # Å — atom-atom cutoff for interface detection
MIN_CONTACTS   = 5
MAX_IMAGE      = 1      # check ±MAX_IMAGE cells each direction
KAPPA_T        = 1.0    # default translational stiffness (k_BT/Å²)
KAPPA_R        = 1.0    # default rotational stiffness (k_BT)
D_MIN_DIFFUSE  = 3.0    # Å — resolution cutoff for diffuse map
TEMPERATURE    = 300.0  # K


In [ ]:
st = gemmi.read_structure(str(PDB_FILE))
st.remove_hydrogens()
st.remove_waters()
st.setup_entities()

cell = st.cell
a1 = np.array(cell.orthogonalize(gemmi.Fractional(1,0,0)).tolist())
a2 = np.array(cell.orthogonalize(gemmi.Fractional(0,1,0)).tolist())
a3 = np.array(cell.orthogonalize(gemmi.Fractional(0,0,1)).tolist())
A_orth  = np.column_stack([a1, a2, a3])        # orthogonalization matrix
B_recip = 2*np.pi * np.linalg.inv(A_orth).T   # reciprocal lattice (columns)

print(f"Cell: {cell.a:.3f}×{cell.b:.3f}×{cell.c:.3f} Å  "
      f"α={cell.alpha:.2f} β={cell.beta:.2f} γ={cell.gamma:.2f}°")
print(f"Space group: {st.spacegroup_hm}   Volume: {cell.volume:.1f} Å³")


## Hybrid Electron Density ($|F_{\rm meas}|$ amplitudes + model phases)


In [ ]:
dc = gemmi.DensityCalculatorX()
dc.d_min = D_MIN_MODEL; dc.rate = RATE_MODEL
dc.set_grid_cell_and_spacegroup(st)
dc.initialize_grid()
dc.add_model_density_to_grid(st[0])
dc.grid.symmetrize_sum()

rho_model   = np.array(dc.grid)
nu_m, nv_m, nw_m = rho_model.shape
F_calc_grid = np.fft.ifftn(rho_model)   # F_calc(h,k,l)/V at [h%nu,k%nv,l%nw]

doc_sf = gemmi.cif.read(str(SF_CIF))
rb     = gemmi.as_refln_blocks(doc_sf)[0]
miller = np.array(rb.make_miller_array())
F_meas = np.array(rb.make_float_array("F_meas_au"))
valid  = np.isfinite(F_meas) & (F_meas > 0)
miller, F_meas = miller[valid], F_meas[valid]

h_i = miller[:,0] % nu_m; k_i = miller[:,1] % nv_m; l_i = miller[:,2] % nw_m
phase_calc = np.angle(F_calc_grid[h_i, k_i, l_i])
F_hybrid   = F_meas * np.exp(1j * phase_calc)

F_grid = np.zeros((nu_m, nv_m, nw_m), dtype=complex)
F_grid[h_i, k_i, l_i] = F_hybrid / cell.volume
F_grid[(-miller[:,0])%nu_m, (-miller[:,1])%nv_m, (-miller[:,2])%nw_m] =     F_hybrid.conj() / cell.volume

rho = np.fft.fftn(F_grid).real.astype(np.float32)
nu, nv, nw = rho.shape
dV = cell.volume / rho.size
print(f"Grid: {nu}×{nv}×{nw}   dV={dV:.4f} Å³   reflections: {valid.sum()}")


In [ ]:
masker = gemmi.SolventMasker(gemmi.AtomicRadiiSet.VanDerWaals)
masker.rprobe = 1.4; masker.rshrink = 0.0
mask_i8 = gemmi.Int8Grid(dc.grid.nu, dc.grid.nv, dc.grid.nw)
mask_i8.unit_cell = dc.grid.unit_cell
mask_i8.spacegroup = dc.grid.spacegroup
masker.put_mask_on_int8_grid(mask_i8, st[0])
mask = np.array(mask_i8).astype(bool)
if rho.shape != mask.shape:
    from scipy.ndimage import zoom
    mask = zoom(mask.astype(float), np.array(rho.shape)/np.array(mask.shape), order=0) > 0.5
rho_masked = rho * mask
print(f"Protein electron content: {rho_masked.sum()*dV:.1f} e⁻")


In [ ]:
ii = np.arange(nu)/nu; jj = np.arange(nv)/nv; kk = np.arange(nw)/nw
Ig, Jg, Kg = np.meshgrid(ii, jj, kk, indexing='ij')
r_xyz = (A_orth @ np.stack([Ig,Jg,Kg]).reshape(3,-1)).reshape(3,nu,nv,nw)

w = rho_masked.ravel()
r_cm = (r_xyz.reshape(3,-1) * w).sum(1) / w.sum()
print(f"Density CM: ({r_cm[0]:.2f}, {r_cm[1]:.2f}, {r_cm[2]:.2f}) Å")


## Coupling Vector $G(\mathbf{q})$

$$G = \begin{pmatrix}G_R\\G_T\end{pmatrix}
    = \begin{pmatrix}i\mathbf{q}\times L(\mathbf{q})\\
                      i\mathbf{q}\,F(\mathbf{q})\end{pmatrix},
\quad F = \int\rho\,e^{-i\mathbf{q}\cdot\mathbf{r}}d^3r,
\quad L = \int(\mathbf{r}-\mathbf{r}_{\rm cm})\rho\,e^{-i\mathbf{q}\cdot\mathbf{r}}d^3r$$

Four FFTs compute $F$ and $L_{x,y,z}$ simultaneously on the full grid.


In [ ]:
rho_w  = rho_masked * dV
F_fft  = np.fft.ifftn(rho_w) * rho_w.size       # F(hkl) at [h%,k%,l%]
dr     = r_xyz - r_cm[:,None,None,None]
L_fft  = np.stack([np.fft.ifftn(dr[ax]*rho_w)*rho_w.size for ax in range(3)])

def G_at_hkl(h, k, l):
    """Return G(q): (6,) complex vector in (Ω,v) ordering."""
    q  = h*B_recip[:,0] + k*B_recip[:,1] + l*B_recip[:,2]
    hi, ki, li = int(h)%nu, int(k)%nv, int(l)%nw
    F  = F_fft[hi, ki, li]
    L  = L_fft[:, hi, ki, li]
    iq = 1j * q
    return np.concatenate([np.cross(iq, L), iq*F])  # (G_R, G_T)

print(f"|F(000)| = {abs(F_fft[0,0,0]):.1f}  (≈ Z_protein)")


## Interface Detection and the Dynamical Matrix

### From BVK + Hessian Symmetry to 3 Unique K Matrices

**Born–von Kármán** (translational invariance) fixes:
$K_{\mathbf{m},\mathbf{m+n}} = K_{\mathbf{0},\mathbf{n}}$ for all $\mathbf{m}$.

**Hessian symmetry** $H_{\mathbf{0n}} = H_{\mathbf{n0}}^T$ combined with BVK gives
$H_{\mathbf{n0}} = H_{\mathbf{0,-n}}$, which forces:

$$K_{-\mathbf{n}} = (A_{-\mathbf{n}}^T)^{-1}\,K_{\mathbf{n}}\,A_{\mathbf{n}}$$

Since $A_{\mathbf{n}} A_{-\mathbf{n}} = I_6$, this makes $K_{-\mathbf{n}}$ *fully determined*
by $K_{\mathbf{n}}$ — so the 6 detected interfaces yield only **3 free K matrices** (21
parameters each). This is a consequence of BVK + Hessian symmetry, *not* an extra
assumption.

### Dynamical Matrix

For contact $\mathbf{n}$ with free spring constant $K_\mathbf{n}$,
and its Hessian-determined partner at $-\mathbf{n}$:

$$D(\mathbf{q}) = \sum_{\rm unique\,\mathbf{n}}
\Bigl[\underbrace{A_\mathbf{n}^T K_\mathbf{n} A_\mathbf{n} + K_\mathbf{n}}_{\text{self}}
- \underbrace{A_\mathbf{n}^T K_\mathbf{n}\,e^{i\mathbf{q}\cdot R_\mathbf{n}}
             + K_\mathbf{n} A_\mathbf{n}\,e^{-i\mathbf{q}\cdot R_\mathbf{n}}}_{\text{cross}}\Bigr]$$

where $A_\mathbf{n} = \mathrm{Ad}_{T_{-R_\mathbf{n}}} = \bigl(\begin{smallmatrix}I&0\\-[R_\mathbf{n}]_\times&I\end{smallmatrix}\bigr)$.

*Acoustic check:* $(A_\mathbf{n}-I)(0,v)=0$ and $(K_\mathbf{n} A_\mathbf{n}-K_\mathbf{n})(0,v)=0$,
so $D(0)(0,v)=0$. ✓


In [ ]:
all_pos = np.array([atom.pos.tolist() for ch in st[0] for res in ch for atom in res])
tree    = cKDTree(all_pos)

# Detect interfaces with periodic images
raw_interfaces = {}
for n_tup in product(range(-MAX_IMAGE,MAX_IMAGE+1), repeat=3):
    if n_tup == (0,0,0): continue
    R_n = sum(n_tup[i]*[a1,a2,a3][i] for i in range(3))
    n_c = sum(len(p) for p in tree.query_ball_point(all_pos + R_n, CONTACT_CUTOFF))
    if n_c >= MIN_CONTACTS:
        raw_interfaces[n_tup] = {'R_n': R_n, 'n_contacts': n_c}

# Canonical unique interface: min(n, -n) lexicographically
def canonical(n):
    return min(n, tuple(-x for x in n))

unique = {}
for n_tup, info in raw_interfaces.items():
    c = canonical(n_tup)
    if c not in unique:
        unique[c] = dict(info, n_tup=n_tup)

print(f"Detected {len(raw_interfaces)} interfaces → {len(unique)} unique K matrices:")
for c, info in sorted(unique.items(), key=lambda x: -x[1]['n_contacts']):
    print(f"  {c}  |R|={np.linalg.norm(info['R_n']):.2f} Å  contacts={info['n_contacts']}")


## K Matrix: Contact Frame and Midpoint Frame

Two natural frames for specifying $K$:

**Contact (body-2) frame:** $\hat{e}_z$ along $R_\mathbf{n}$, origin at body-2 CM.
Standard choice; gives $B^T = (-\mathrm{Ad}_{g_{12}^{-1}}, I_6)$.

**Midpoint frame:** origin at $R_\mathbf{n}/2$, same orientation. Both bodies are at
$\pm R_\mathbf{n}/2$, making $B^T = (-\mathrm{Ad}_{T_{R_\mathbf{n}/2}}, \mathrm{Ad}_{T_{-R_\mathbf{n}/2}})$
— symmetric in bodies 1 and 2. The K matrices are related by a center-of-stiffness
shift $\hat{c} = -R_\mathbf{n}/2$:

$$K_{mid} = \Gamma_{-R_\mathbf{n}/2}^{-T}\,K_{\rm contact}\,\Gamma_{-R_\mathbf{n}/2}^{-1},
\quad \Gamma_c = \mathrm{diag}(I, I, I,\; I, I, I)\text{ with off-diagonal }[c]_\times$$

Set `USE_MIDPOINT_FRAME = True` to input K in the midpoint frame (recommended for
physical interpretation); `False` for the body-2/contact frame. Both are transformed
to the lab frame before building $D(q)$.


In [ ]:
USE_MIDPOINT_FRAME = True   # True: input K in midpoint frame; False: body-2 frame

def skew(v):
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])

def contact_frame(R_n):
    """Rotation matrix: columns = (ex, ey, ez), ez along R_n."""
    ez = R_n / np.linalg.norm(R_n)
    perp = np.array([1.,0.,0.]) if abs(ez[0]) < 0.9 else np.array([0.,1.,0.])
    ex = np.cross(perp, ez); ex /= np.linalg.norm(ex)
    return np.column_stack([ex, np.cross(ez,ex), ez])

def shift_origin(K, c):
    """Shift K by translation c: K' = Gamma_c^{-T} K Gamma_c^{-1}."""
    # Gamma_c in (Omega,v) basis: [[I,0],[c×,I]]
    # Gamma_c^{-1} = [[I,0],[-c×,I]]
    Gc_inv = np.eye(6); Gc_inv[3:6,0:3] = -skew(c)
    return Gc_inv.T @ K @ Gc_inv

def K_frame_to_lab(K_in_frame, R_frame, R_n, use_midpoint):
    """Transform K from contact/midpoint frame to lab frame."""
    if use_midpoint:
        # midpoint frame: shift origin by +R_n/2 to get to body-2 frame first
        K_body2 = shift_origin(K_in_frame, R_n/2)
    else:
        K_body2 = K_in_frame
    # Then rotate from contact orientation to lab orientation
    Gamma = np.block([[R_frame, np.zeros((3,3))],[np.zeros((3,3)), R_frame]])
    return Gamma @ K_body2 @ Gamma.T

# ── User input: K in the chosen frame ──────────────────────────────────────────
# Default: isotropic, decoupled, slightly stiffer normal to interface
def default_K(kappa_T=KAPPA_T, kappa_R=KAPPA_R, anisotropy=1.5):
    K = np.zeros((6,6))
    K[0:3,0:3] = kappa_R * np.eye(3)
    K[3:6,3:6] = kappa_T * np.diag([1., 1., anisotropy])
    return K

# K_input_dict: override any entry with a custom 6×6 symmetric matrix
K_input_dict = {c: default_K() for c in unique}

# ── Transform to lab frame ──────────────────────────────────────────────────────
for c, info in unique.items():
    info['R_frame'] = contact_frame(info['R_n'])

K_lab = {c: K_frame_to_lab(K_input_dict[c], unique[c]['R_frame'],
                             unique[c]['R_n'], USE_MIDPOINT_FRAME)
          for c in unique}
print("K matrices transformed to lab frame. K_RR diag, K_TT diag per interface:")
for c in unique:
    print(f"  {c}: K_RR={np.diag(K_lab[c][0:3,0:3]).round(3)}"
          f"  K_TT={np.diag(K_lab[c][3:6,3:6]).round(3)}")


In [ ]:
ATOMIC_MASS = {'C':12.011,'N':14.007,'O':15.999,'S':32.06,'P':30.974,
               'SE':78.96,'H':1.008,'FE':55.845,'ZN':65.38,'CA':40.078}
masses, apos = [], []
for ch in st[0]:
    for res in ch:
        for atom in res:
            masses.append(ATOMIC_MASS.get(atom.element.name.upper(), 12.0))
            apos.append(atom.pos.tolist())
masses = np.array(masses); apos = np.array(apos)
m_total  = masses.sum()
r_cm_at  = (masses[:,None]*apos).sum(0)/m_total
dr_a     = apos - r_cm_at
r2       = (dr_a**2).sum(1)
J        = (masses[:,None,None] * (r2[:,None,None]*np.eye(3)[None]
           - dr_a[:,:,None]*dr_a[:,None,:])).sum(0)
M_mat    = np.block([[J, np.zeros((3,3))],[np.zeros((3,3)), m_total*np.eye(3)]])

import scipy.constants as const
kBT_SI   = const.k * TEMPERATURE
omega_unit = np.sqrt(kBT_SI / (const.atomic_mass * (1e-10)**2))  # rad/s
freq_unit  = omega_unit / (2*np.pi) / 1e12                         # THz
print(f"Total mass: {m_total:.0f} amu")
print(f"Frequency unit: 1 (k_BT/amu/Å²)^½ = {freq_unit:.3f} THz")


In [ ]:
def ad_translation(R_n):
    """Ad_{T_{-R_n}}: 6×6 in (Omega,v) order; lower-left = -R_n×."""
    A = np.eye(6); A[3:6,0:3] = -skew(R_n); return A

def dynamical_matrix(q_cart, unique, K_lab):
    """
    Correct D(q) using BVK + Hessian symmetry.
    Each unique interface n contributes BOTH the n and -n contacts.
    K_{-n} is determined by K_n, giving:
        self(-n)  = K_n          (since A_n A_{-n} = I_6)
        cross(-n) = K_n A_n e^{-iq·R_n}
    """
    D = np.zeros((6,6), dtype=complex)
    for c, info in unique.items():
        R_n = info['R_n']
        K   = K_lab[c]
        A_n = ad_translation(R_n)
        pn   = np.exp( 1j * q_cart @ R_n)
        pneg = pn.conj()
        # Contact n
        D += A_n.T @ K @ A_n          # self(n)
        D -= (A_n.T @ K) * pn         # cross(n)
        # Contact -n  [K_{-n} determined by Hessian symmetry]
        D += K                         # self(-n)
        D -= (K @ A_n) * pneg         # cross(-n)
    return D

# Sanity: 3 eigenvalues should be ≈ 0 at Gamma
Msq     = np.linalg.cholesky(M_mat)
Msq_inv = np.linalg.inv(Msq)
D0      = dynamical_matrix(np.zeros(3), unique, K_lab)
ev0     = np.linalg.eigvalsh((Msq_inv @ D0 @ Msq_inv.T).real)
print("D(q=0) M-weighted eigenvalues (3 should be ≈ 0):", ev0.round(6))


In [ ]:
HSP = {'Γ':np.array([0.,0.,0.]),'X':np.array([.5,0.,0.]),
       'Y':np.array([0.,.5,0.]),'Z':np.array([0.,0.,.5])}
path_labels = ['Γ','X','Γ','Y','Γ','Z','Γ']
N_seg = 60

q_frac, tick_idx = [], [0]
for seg in range(len(path_labels)-1):
    p0, p1 = HSP[path_labels[seg]], HSP[path_labels[seg+1]]
    last = (seg == len(path_labels)-2)
    for t in np.linspace(0, 1, N_seg, endpoint=last):
        q_frac.append(p0*(1-t)+p1*t)
    if not last: tick_idx.append(len(q_frac))
tick_idx.append(len(q_frac)-1)

q_frac  = np.array(q_frac)
q_cart  = (B_recip @ q_frac.T).T
freqs_bs = []
for q in q_cart:
    D  = dynamical_matrix(q, unique, K_lab)
    ev = np.linalg.eigvalsh((Msq_inv @ D @ Msq_inv.T).real)
    freqs_bs.append(np.sqrt(np.maximum(ev,0)) * freq_unit)
freqs_bs = np.array(freqs_bs)

x = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(q_cart,axis=0),axis=1))])
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(x, freqs_bs, color='steelblue', lw=1.2)
for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
ax.set_ylabel('Frequency (THz)'); ax.set_xlim(x[0],x[-1]); ax.set_ylim(0)
ax.set_title('Rigid-Body Phonon Band Structure — 6O2H (P1)')
plt.tight_layout(); plt.savefig('band_structure.png', dpi=150); plt.show()


In [ ]:
h_max = int(np.ceil(cell.a/D_MIN_DIFFUSE))+1
k_max = int(np.ceil(cell.b/D_MIN_DIFFUSE))+1
L_LAYER = 0
print(f"Computing I(h,k,{L_LAYER}): h∈[{-h_max},{h_max}], k∈[{-k_max},{k_max}]...")

I_map = np.zeros((2*h_max+1, 2*k_max+1))
for ih, h in enumerate(range(-h_max, h_max+1)):
    for ik, k in enumerate(range(-k_max, k_max+1)):
        q = h*B_recip[:,0] + k*B_recip[:,1] + L_LAYER*B_recip[:,2]
        if np.linalg.norm(q) < 1e-6 or np.linalg.norm(q) > 2*np.pi/D_MIN_DIFFUSE:
            continue
        G = G_at_hkl(h, k, L_LAYER)
        D = dynamical_matrix(q, unique, K_lab)
        try:
            I_map[ih,ik] = np.real(G.conj() @ np.linalg.solve(D, G))
        except np.linalg.LinAlgError:
            pass

fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(np.log1p(np.abs(I_map)).T, origin='lower', cmap='inferno',
               extent=[-h_max,h_max,-k_max,k_max])
plt.colorbar(im, ax=ax, label='log(1+|I|)')
ax.set_xlabel('h'); ax.set_ylabel('k')
ax.set_title(f'Diffuse scattering I(h,k,{L_LAYER}) — log scale')
plt.tight_layout(); plt.savefig('diffuse_map_hk0.png', dpi=150); plt.show()


## In-Notebook Mode Visualization (Surface Representation)

We use **nglview** to render a molecular surface directly in the notebook,
and display each phonon mode as a smooth animation over a morph trajectory.

Each mode at the chosen wavevector $\mathbf{q}$ displaces every atom by:
$$\delta\mathbf{r}_a = A\bigl[v_s + \Omega_s\times(\mathbf{r}_a-\mathbf{r}_{\rm cm})\bigr]$$
where $A$ is the (real) animation amplitude and $(\Omega_s, v_s)$ are the mode
eigenvector components.


In [ ]:
# Install nglview if absent (runs once; restart kernel if prompted)
try:
    import nglview as nv
    print("nglview available:", nv.__version__)
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'nglview'], check=True)
    import nglview as nv
    print("nglview installed:", nv.__version__)


In [ ]:
from scipy.spatial.transform import Rotation as Rot
import tempfile, os

Q_VIZ   = np.array([0.5, 0.0, 0.0])   # fractional — change as desired
AMPLITUDE = 3.0   # Å exaggeration
N_FRAMES  = 24    # animation frames

q_viz  = B_recip @ Q_VIZ
D_viz  = dynamical_matrix(q_viz, unique, K_lab)
ev_v, evec_v = np.linalg.eigh((Msq_inv @ D_viz @ Msq_inv.T).real)
ev_v         = np.maximum(ev_v, 0)
freqs_viz    = np.sqrt(ev_v) * freq_unit
evecs_lab    = Msq_inv.T @ evec_v        # (6,6) columns = lab-frame eigenvectors

print(f"Modes at q={Q_VIZ}:")
for s in range(6):
    Om, v = evecs_lab[:3,s].real, evecs_lab[3:,s].real
    print(f"  {s}: {freqs_viz[s]:.4f} THz  |Ω|={np.linalg.norm(Om):.3f}  |v|={np.linalg.norm(v):.3f}")


In [ ]:
def pdb_frame(apos, atom_records):
    """Minimal PDB string from positions array and pre-built record headers."""
    lines = []
    for i, (hdr, p) in enumerate(zip(atom_records, apos)):
        lines.append(f"{hdr}{p[0]:8.3f}{p[1]:8.3f}{p[2]:8.3f}  1.00  0.00\n")
    lines.append("END\n")
    return "".join(lines)

# Pre-build PDB header strings for each atom (everything except xyz)
atom_records = []
idx = 0
for ch in st[0]:
    for res in ch:
        for atom in res:
            idx += 1
            name = atom.name.ljust(4) if len(atom.name)<4 else atom.name
            atom_records.append(
                f"ATOM  {idx:5d} {name:<4s} {res.name:3s} {ch.name}"
                f"{res.seqid.num:4d}    "
            )

base_pos = apos.copy()   # (N_atoms, 3) from mass-matrix cell

def mode_frames(mode_idx, amplitude=AMPLITUDE, n_frames=N_FRAMES):
    """Return list of PDB strings for one mode (one sine cycle)."""
    Om = evecs_lab[:3, mode_idx].real
    v  = evecs_lab[3:, mode_idx].real
    norm = np.sqrt((Om**2).sum() + (v**2).sum())
    if norm < 1e-10: return [pdb_frame(base_pos, atom_records)]*n_frames
    Om /= norm; v /= norm
    frames = []
    for f in range(n_frames):
        scale = amplitude * np.sin(2*np.pi*f/n_frames)
        Om_s  = scale * Om
        v_s   = scale * v
        if np.linalg.norm(Om_s) > 1e-12:
            new_pos = Rot.from_rotvec(Om_s).apply(base_pos - r_cm_at) + r_cm_at + v_s
        else:
            new_pos = base_pos + v_s
        frames.append(pdb_frame(new_pos, atom_records))
    return frames

print("Frame generation ready. Run the cells below to view each mode.")


In [ ]:
# ── View a single mode ─────────────────────────────────────────────────────────
# Change MODE_IDX (0–5) to select the mode; re-run cell to update.
MODE_IDX = 0

frames = mode_frames(MODE_IDX)

# Write frames to a temporary multi-model PDB
tmp = tempfile.NamedTemporaryFile(suffix='.pdb', delete=False, mode='w')
for i, frame in enumerate(frames):
    tmp.write(f"MODEL     {i+1}\n")
    tmp.write(frame.replace("\nEND\n", "\nENDMDL\n"))
tmp.write("END\n")
tmp.close()

# Display with nglview: surface + ball-and-stick
view = nv.show_file(tmp.name, ext='pdb')
view.clear_representations()
view.add_surface(selection='protein', opacity=0.7, color='skyblue')
view.add_ball_and_stick(selection='protein', opacity=0.2)
view.player.parameters = dict(delay=80, step=1)   # ~12 fps
view._remote_call('autoView', target='Widget')
print(f"Mode {MODE_IDX}: {freqs_viz[MODE_IDX]:.4f} THz  "
      f"|Ω|={np.linalg.norm(evecs_lab[:3,MODE_IDX]):.3f}  "
      f"|v|={np.linalg.norm(evecs_lab[3:,MODE_IDX]):.3f}")
view


In [ ]:
# ── View all 6 modes as a grid ─────────────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

views = []
for s in range(6):
    frms = mode_frames(s)
    tmp = tempfile.NamedTemporaryFile(suffix='.pdb', delete=False, mode='w')
    for i, frame in enumerate(frms):
        tmp.write(f"MODEL     {i+1}\n")
        tmp.write(frame.replace("\nEND\n", "\nENDMDL\n"))
    tmp.write("END\n"); tmp.close()
    v = nv.show_file(tmp.name, ext='pdb')
    v.clear_representations()
    v.add_surface(selection='protein', opacity=0.75, color=['skyblue','salmon','lightgreen',
                                                             'gold','plum','coral'][s])
    v.player.parameters = dict(delay=80, step=1)
    v._remote_call('autoView', target='Widget')
    v.layout = widgets.Layout(width='300px', height='300px')
    views.append(v)

labels = [widgets.Label(f"Mode {s}: {freqs_viz[s]:.3f} THz") for s in range(6)]
grid = widgets.GridBox(
    [widgets.VBox([labels[s], views[s]]) for s in range(6)],
    layout=widgets.Layout(grid_template_columns="repeat(3, 320px)")
)
display(grid)


## Summary

| Output | File |
|--------|------|
| Band structure | `band_structure.png` |
| Diffuse map (hk0) | `diffuse_map_hk0.png` |
| In-notebook animation | nglview widgets above |

**Fitting K to data:** minimise $\sum_\mathbf{q}[I_{\rm obs}(\mathbf{q})-I_{\rm calc}(\mathbf{q})]^2$
over the $3\times 21=63$ parameters using $\nabla_K I = -G^\dagger D^{-1}(\partial D/\partial K)D^{-1}G$.

**Frame choices for K:**
- *Midpoint frame* (`USE_MIDPOINT_FRAME=True`): symmetric in the two bodies;
  $K_{TT}^{zz}$ = stiffness pushing the two molecules apart,
  $K_{RR}$ = librational coupling at the interface.
- *Body-2/contact frame* (`False`): $B^T=(-\mathrm{Ad}_{g_{12}^{-1}}, I_6)$, natural for deriving $D(q)$.
- Both give identical physics; the lab-frame $D(q)$ is frame-independent.
